# M3 评测（v20）——checkpoint-016000（真实 16k updates）× 100 episodes

In [ ]:
# ===== v20 M3 评测：checkpoint-016000（真实 16k updates）× 100 episodes 协议评测 =====
# 设计原则：训练是前台进程，任何崩溃的 traceback 直接出现在批处理日志里，无静默失败。
# 无隧道/sshd/watchdog——监控走 ntfy 信标（topic smolvla-ep-b7Vq2mN4tX）。
import os, re, glob, shutil, time, subprocess, urllib.request

W = "/kaggle/working"
TOPIC = "smolvla2-ep-Kp9wQ4tRz7"
CODE_SHA = "146d8474932c8b566717eb2738ba459337ab71c4"
REV = "a1aaacb7f6cd6ee5fb43120f673cebb0cfea7dd4"
HUB_SHA = "5e8d12a6e2975b0e5e5fce7c8caf47c371d257b6"
os.environ["HF_HOME"] = W + "/hf_cache"
os.makedirs(W + "/logs", exist_ok=True)

def run(cmd, timeout=1800):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout)
    return (r.stdout + r.stderr).strip()

def beacon(msg):
    try:
        urllib.request.urlopen(urllib.request.Request("https://ntfy.sh/" + TOPIC, data=str(msg).encode()), timeout=8)
    except Exception as e:
        print("[beacon fail]", repr(e), flush=True)

# ---- 1) lerobot 安装（固定 commit）+ dtype 补丁 ----
def have_lerobot():
    return "0.6.2" in run("python -c 'import lerobot; print(lerobot.__version__)'")
if not have_lerobot():
    for cmd in ('uv pip install --system "lerobot[smolvla,libero] @ git+https://github.com/huggingface/lerobot.git@' + CODE_SHA + '"',
                'pip install --no-input "lerobot[smolvla,libero] @ git+https://github.com/huggingface/lerobot.git@' + CODE_SHA + '"'):
        print("[1] install ...", flush=True)
        print(run(cmd, timeout=1200)[-600:], flush=True)
        if have_lerobot():
            break
    if not have_lerobot():
        beacon("FAIL-install")
        raise RuntimeError("lerobot 安装失败（两种方式均失败，输出见上）")
run('sed -i \'s/torch_dtype="bfloat16"/torch_dtype="float32"/\' /usr/local/lib/python3.13/dist-packages/lerobot/policies/smolvla/smolvlm_with_expert.py')
ok = "float32" in run("grep torch_dtype /usr/local/lib/python3.13/dist-packages/lerobot/policies/smolvla/smolvlm_with_expert.py")
print("[1] lerobot 0.6.2 + dtype patch:", ok, flush=True)
if not ok:
    beacon("FAIL-dtype")
    raise RuntimeError("dtype 补丁失败")
beacon("OK-install")

# ---- 2) 从 HF 拉取 16k checkpoint（checkpoints-backup/0008000/pretrained_model）----
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # Add-ons -> Secrets 里添加
from huggingface_hub import snapshot_download
CKPT = snapshot_download("A2bao/smolvla-libero-spatial",
                         allow_patterns=["checkpoints-backup/0008000/*"],
                         local_dir=W + "/ckpt_dl") + "/checkpoints-backup/0008000/pretrained_model"
print("[2] checkpoint:", CKPT, flush=True)
beacon("OK-ckpt")

# ---- 3) 训练产物自检（权重 + processor 配置齐全）----
for f in ("model.safetensors", "policy_preprocessor.json", "train_config.json"):
    if not os.path.exists(CKPT + "/" + f):
        beacon("FAIL-ckpt-incomplete")
        raise RuntimeError("checkpoint 缺文件: " + f)
print("[3] checkpoint 自检通过", flush=True)

# ---- 4) assets + libero 配置（失败仅影响训练内评测，不致命）----
ASSETS = """import time
from pathlib import Path
from huggingface_hub import snapshot_download
import importlib.util
home = Path.home()
t0 = time.time()
snapshot_download(repo_id='lerobot/libero-assets', repo_type='dataset', local_dir='/kaggle/working/libero_assets')
spec = importlib.util.find_spec('libero')
lib_dir = Path(spec.origin).parent / 'libero'
(home / '.libero').mkdir(exist_ok=True)
lines = [
    'assets: /kaggle/working/libero_assets',
    'bddl_files: %s/bddl_files' % lib_dir,
    'datasets: %s/../datasets' % lib_dir.parent,
    'init_states: %s/init_files' % lib_dir,
]
(home / '.libero' / 'config.yaml').write_text(chr(10).join(lines) + chr(10))
print('assets %.0fs' % (time.time() - t0))
"""
open("/tmp/assets.py", "w").write(ASSETS)
out = run("python /tmp/assets.py", timeout=1800)
print("[3] assets:", out[-150:], flush=True)
if "assets" not in out:
    print("[3][warn] 资产失败，训练内评测会失败", flush=True)

# ---- 5) 数据集缓存修复（pinned revision，3 次重试）----
open("/tmp/dsfix.py", "w").write(
    "from huggingface_hub import snapshot_download\n"
    "snapshot_download('lerobot/libero', repo_type='dataset', revision='" + REV + "', max_workers=4)\n"
    "print('DS_OK')\n")
for attempt in range(3):
    out = run("python /tmp/dsfix.py 2>&1 | tail -2", timeout=2400)
    if "DS_OK" in out:
        break
    print("[5] 数据集修复重试", attempt + 1, ":", out[-200:], flush=True)
    time.sleep(20)
else:
    beacon("FAIL-dataset")
    raise RuntimeError("数据集缓存修复失败: " + out[-300:])
print("[5] 数据集缓存完整（revision " + REV[:8] + "）", flush=True)
beacon("OK-setup")

# ---- 6) 前台评测：输出直写文件（不经 kernel 显示通道，杜绝反压卡死）----
import threading
ENV = ("export PATH=/opt/bin:$PATH; export LD_LIBRARY_PATH=/usr/local/nvidia/lib64:$LD_LIBRARY_PATH; "
       "export MUJOCO_GL=egl; export HF_HOME=" + W + "/hf_cache; export PYTHONUNBUFFERED=1;")
CMD = ("lerobot-eval --policy.path=" + CKPT + " "
       "--env.type=libero --env.task=libero_spatial "
       "--env.observation_height=256 --env.observation_width=256 "
       "--env.init_states=true --env.hard_reset=true "
       "--policy.n_action_steps=1 "
       "--eval.n_episodes=10 --eval.batch_size=1")
LOG = W + "/logs/eval.log"
os.makedirs(W + "/logs", exist_ok=True)
open(LOG, "w").close()

# 监视线程：每 90s 推一次进度（已完成 episode 数 + 日志最新一行）
_done = W + "/logs/eval.done"
def _watch():
    while not os.path.exists(_done):
        time.sleep(90)
        try:
            raw = open(LOG, errors="ignore").read().replace("\r", "\n")
            eps = len(re.findall(r"Running rollout", raw))
            lines = [l for l in raw.split("\n") if l.strip()]
            last = lines[-1][-90:] if lines else "(empty)"
            beacon("EP%d | %s" % (eps, last))
        except Exception as e:
            print("[watch]", repr(e), flush=True)
threading.Thread(target=_watch, daemon=True).start()

MAX_TRY = 2
for attempt in range(1, MAX_TRY + 1):
    print("[6] 评测尝试 #%d —— 输出直写文件，画面不会滚动，进度看 ntfy 信标" % attempt, flush=True)
    ret = subprocess.call("bash -c '" + ENV + " " + CMD + " > " + LOG + " 2>&1'", shell=True)
    open(_done, "w").close()
    tail = run("tail -c 2000 " + LOG + " | tr '\\r' '\n'")
    print(tail[-1200:], flush=True)
    if ret == 0:
        m = re.search(r"(?:Success rate|success rate)[:\s]*([0-9.]+)", tail)
        rate = m.group(1) if m else "see-log"
        beacon("EVAL-DONE rate=" + str(rate))
        print("[6] 评测完成，成功率:", rate, flush=True)
        break
    beacon("EVAL-FAIL try%d ret=%d" % (attempt, ret))
    print("[6] 评测异常 ret=%d" % ret, flush=True)
    time.sleep(20)
else:
    beacon("EVAL-GIVEUP")
    raise RuntimeError("评测连续失败，详见 logs/eval.log")
print("ALL DONE", flush=True)
